# Exploratory Data Analysis (EDA)

Uses `data/cleaned_data.csv`. Visualizations use **matplotlib only** (no seaborn), per project requirements.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if (Path.cwd().name == "notebooks") else Path.cwd()
IMG = ROOT / "images"
IMG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(ROOT / "data" / "cleaned_data.csv", parse_dates=["Date"])
df.head()

## Correlation heatmap (numeric features)
Pairwise linear relationships among Revenue, Profit, Quantity, and derived margin.

In [ ]:
num = df[["Revenue", "Profit", "Quantity", "Profit_Margin_Pct"]].dropna()
corr = num.corr()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.values, cmap="RdYlGn", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", color="black", fontsize=9)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Correlation heatmap (numeric KPI drivers)")
plt.tight_layout()
fig.savefig(IMG / "correlation_heatmap.png", dpi=150)
plt.show()

## Revenue trend (monthly)
Aggregate revenue by calendar month to see growth and seasonality.

In [ ]:
df["YearMonth"] = df["Date"].dt.to_period("M").dt.to_timestamp()
monthly = df.groupby("YearMonth", as_index=False)["Revenue"].sum()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly["YearMonth"], monthly["Revenue"], color="#1f77b4", linewidth=2)
ax.set_title("Total revenue by month")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Segment overview — revenue by category
Which product categories drive the top line?

In [ ]:
cat_rev = df.groupby("Category", as_index=False)["Revenue"].sum().sort_values("Revenue", ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(cat_rev["Category"], cat_rev["Revenue"], color="#2ca02c")
ax.set_title("Revenue by category")
ax.set_ylabel("Revenue")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()